In [0]:
CREATE OR REPLACE TEMP VIEW silver_ads
AS
SELECT *
FROM parquet.`abfss://silver@marketingde2026.dfs.core.windows.net/amazon/amazon_ads.parquet`;

In [0]:
SELECT
    campaign_id,
    campaign_name,
    brand,
    product,
    SUM(impressions) AS total_impressions,
    SUM(clicks) AS total_clicks,
    ROUND(SUM(spend), 2) AS total_spend,
    SUM(orders) AS total_orders,
    ROUND(SUM(sales), 2) AS total_sales
FROM silver_ads
GROUP BY
    campaign_id,
    campaign_name,
    brand,
    product
ORDER BY total_sales DESC;

campaign_id,campaign_name,brand,product,total_impressions,total_clicks,total_spend,total_orders,total_sales
AMZ001,Classmate_Search,Classmate,Notebook,4131357,128723.0,2357406.0,9218,6542354.0
AMZ004,Stationery_Generic,Classmate,Stationery,3742483,115743.0,2336032.0,8421,6172022.0
AMZ002,Classmate_Display,Classmate,Pens,3915283,121874.0,2560171.0,9650,5760596.0
AMZ003,Verite_Launch,Classmate,Verite Notebook,4225748,131301.0,2207325.0,8698,5421406.0


In [0]:
SELECT
    campaign_id,
    campaign_name,
    brand,
    product,
    SUM(impressions) AS total_impressions,
    SUM(clicks) AS total_clicks,
    ROUND(SUM(spend), 2) AS total_spend,
    SUM(orders) AS total_orders,
    ROUND(SUM(sales), 2) AS total_sales,

    ROUND(
        (SUM(clicks) / SUM(impressions)) * 100,
        2
    ) AS ctr,

    ROUND(
        SUM(spend) / NULLIF(SUM(clicks), 0),
        2
    ) AS cpc,

    ROUND(
        (SUM(orders) / NULLIF(SUM(clicks), 0)) * 100,
        2
    ) AS conversion_rate,

    ROUND(
        SUM(sales) / NULLIF(SUM(spend), 0),
        2
    ) AS roas

FROM silver_ads

GROUP BY
    campaign_id,
    campaign_name,
    brand,
    product

ORDER BY roas DESC;

campaign_id,campaign_name,brand,product,total_impressions,total_clicks,total_spend,total_orders,total_sales,ctr,cpc,conversion_rate,roas
AMZ001,Classmate_Search,Classmate,Notebook,4131357,128723.0,2357406.0,9218,6542354.0,3.12,18.31,7.16,2.78
AMZ004,Stationery_Generic,Classmate,Stationery,3742483,115743.0,2336032.0,8421,6172022.0,3.09,20.18,7.28,2.64
AMZ003,Verite_Launch,Classmate,Verite Notebook,4225748,131301.0,2207325.0,8698,5421406.0,3.11,16.81,6.62,2.46
AMZ002,Classmate_Display,Classmate,Pens,3915283,121874.0,2560171.0,9650,5760596.0,3.11,21.01,7.92,2.25


In [0]:
SELECT
    campaign_id,
    campaign_name,
    brand,
    product,

    SUM(impressions) AS total_impressions,
    SUM(clicks) AS total_clicks,
    ROUND(SUM(spend), 2) AS total_spend,
    SUM(orders) AS total_orders,
    ROUND(SUM(sales), 2) AS total_sales,

    ROUND(
        SUM(clicks) / SUM(impressions) * 100,
        2
    ) AS ctr,

    ROUND(
        SUM(spend) / SUM(clicks),
        2
    ) AS cpc,

    ROUND(
        SUM(sales) / SUM(spend),
        2
    ) AS roas,

    ROUND(
        SUM(sales) / SUM(orders),
        2
    ) AS aov

FROM silver_ads

GROUP BY
    campaign_id,
    campaign_name,
    brand,
    product

ORDER BY roas DESC;

campaign_id,campaign_name,brand,product,total_impressions,total_clicks,total_spend,total_orders,total_sales,ctr,cpc,roas,aov
AMZ001,Classmate_Search,Classmate,Notebook,4131357,128723.0,2357406.0,9218,6542354.0,3.12,18.31,2.78,709.74
AMZ004,Stationery_Generic,Classmate,Stationery,3742483,115743.0,2336032.0,8421,6172022.0,3.09,20.18,2.64,732.93
AMZ003,Verite_Launch,Classmate,Verite Notebook,4225748,131301.0,2207325.0,8698,5421406.0,3.11,16.81,2.46,623.29
AMZ002,Classmate_Display,Classmate,Pens,3915283,121874.0,2560171.0,9650,5760596.0,3.11,21.01,2.25,596.95


In [0]:
INSERT OVERWRITE TABLE marketing_gold_campaign_performance
SELECT
    campaign_id,
    campaign_name,
    brand,
    product,

    SUM(impressions) AS total_impressions,
    SUM(clicks) AS total_clicks,
    ROUND(SUM(spend), 2) AS total_spend,
    SUM(orders) AS total_orders,
    ROUND(SUM(sales), 2) AS total_sales,

    ROUND(SUM(clicks) / SUM(impressions) * 100, 2) AS ctr,
    ROUND(SUM(spend) / SUM(clicks), 2) AS cpc,
    ROUND(SUM(sales) / SUM(spend), 2) AS roas,
    ROUND(SUM(sales) / SUM(orders), 2) AS aov

FROM silver_ads

GROUP BY
    campaign_id,
    campaign_name,
    brand,
    product;

num_affected_rows,num_inserted_rows
4,4


In [0]:
SELECT *
FROM marketing_gold_campaign_performance
ORDER BY roas DESC;

campaign_id,campaign_name,brand,product,total_impressions,total_clicks,total_spend,total_orders,total_sales,ctr,cpc,roas,aov
AMZ001,Classmate_Search,Classmate,Notebook,4131357,128723.0,2357406.0,9218,6542354.0,3.12,18.31,2.78,709.74
AMZ004,Stationery_Generic,Classmate,Stationery,3742483,115743.0,2336032.0,8421,6172022.0,3.09,20.18,2.64,732.93
AMZ003,Verite_Launch,Classmate,Verite Notebook,4225748,131301.0,2207325.0,8698,5421406.0,3.11,16.81,2.46,623.29
AMZ002,Classmate_Display,Classmate,Pens,3915283,121874.0,2560171.0,9650,5760596.0,3.11,21.01,2.25,596.95


In [0]:
SELECT
    COUNT(*) AS total_campaigns,
    SUM(total_impressions) AS total_impressions,
    SUM(total_clicks) AS total_clicks,
    ROUND(SUM(total_spend), 2) AS total_spend,
    SUM(total_orders) AS total_orders,
    ROUND(SUM(total_sales), 2) AS total_sales,
    ROUND(SUM(total_sales) / SUM(total_spend), 2) AS overall_roas
FROM marketing_gold_campaign_performance;

total_campaigns,total_impressions,total_clicks,total_spend,total_orders,total_sales,overall_roas
4,16014871,497641.0,9460934.0,35987,2.3896378E7,2.53


In [0]:
SELECT
    campaign_id,
    campaign_name,
    product,
    total_impressions,
    total_clicks,
    total_spend,
    total_orders,
    total_sales,
    ctr,
    cpc,
    roas,
    aov
FROM marketing_gold_campaign_performance
ORDER BY roas DESC;

campaign_id,campaign_name,product,total_impressions,total_clicks,total_spend,total_orders,total_sales,ctr,cpc,roas,aov
AMZ001,Classmate_Search,Notebook,4131357,128723.0,2357406.0,9218,6542354.0,3.12,18.31,2.78,709.74
AMZ004,Stationery_Generic,Stationery,3742483,115743.0,2336032.0,8421,6172022.0,3.09,20.18,2.64,732.93
AMZ003,Verite_Launch,Verite Notebook,4225748,131301.0,2207325.0,8698,5421406.0,3.11,16.81,2.46,623.29
AMZ002,Classmate_Display,Pens,3915283,121874.0,2560171.0,9650,5760596.0,3.11,21.01,2.25,596.95


In [0]:
SELECT
    COUNT(*) AS total_campaigns,
    SUM(total_impressions) AS total_impressions,
    SUM(total_clicks) AS total_clicks,
    ROUND(SUM(total_spend), 2) AS total_spend,
    SUM(total_orders) AS total_orders,
    ROUND(SUM(total_sales), 2) AS total_sales,
    ROUND(SUM(total_clicks) / SUM(total_impressions) * 100, 2) AS overall_ctr,
    ROUND(SUM(total_spend) / SUM(total_clicks), 2) AS overall_cpc,
    ROUND(SUM(total_sales) / SUM(total_spend), 2) AS overall_roas,
    ROUND(SUM(total_sales) / SUM(total_orders), 2) AS overall_aov
FROM marketing_gold_campaign_performance;

total_campaigns,total_impressions,total_clicks,total_spend,total_orders,total_sales,overall_ctr,overall_cpc,overall_roas,overall_aov
4,16014871,497641.0,9460934.0,35987,2.3896378E7,3.11,19.01,2.53,664.03
